# ============================================================
# STEP 2E— FEATURE SELECTION PLOTS
# Cluster Computing journal standard
# ============================================================
# Produces 12 publication-ready figures covering:
#   1.  Correlation heatmap (top 30 by variance, clean dedup file)
#   2.  Top-25 |Pearson corr| with label — binary + 5class
#   3.  Near-zero variance analysis
#   4.  Mutual information scores — binary + 5class
#   5.  Multi-method consensus ranking (MI + |corr| + variance)
#   6.  Feature stability across 5 CV folds (std bars)
#   7.  Feature selection curve (F1 vs top-k features)
#   8.  Feature group importance breakdown
#   9.  Pairwise redundancy — correlated feature pairs
#  10.  Class-conditional distribution (top-6 features, violin)
#  11.  Feature redundancy network (graph: |r|>0.7 edges)
#  12.  Comprehensive selection summary table (LaTeX)
# ============================================================

In [ ]:


import os, warnings
import numpy as np
import pandas as pd
import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import matplotlib.patches as mpatches
from matplotlib.lines import Line2D
from scipy.cluster import hierarchy
from scipy.spatial.distance import squareform
from sklearn.feature_selection import mutual_info_classif
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.inspection import permutation_importance
from sklearn.metrics import f1_score
from sklearn.preprocessing import LabelEncoder

warnings.filterwarnings("ignore")
SEED = 42
np.random.seed(SEED)

# ── CONFIG ────────────────────────────────────────────────────
DATA_DIR   = "/content/drive/MyDrive/Colab Notebooks/Research/New_Dataset_May"
INPUT_FILE = os.path.join(DATA_DIR, "features_full_final_inductive_dedup.csv")
OUT_DIR    = os.path.join(DATA_DIR, "journal_eval_feature_selection")
PNG_DIR    = os.path.join(OUT_DIR, "figures_png")
PDF_DIR    = os.path.join(OUT_DIR, "figures_pdf")
TEX_DIR    = os.path.join(OUT_DIR, "tex")
for d in [OUT_DIR, PNG_DIR, PDF_DIR, TEX_DIR]:
    os.makedirs(d, exist_ok=True)

LEAKY = {"is_tr_domain","is_https","cluster_malicious_ratio","campaign_token_score"}
META  = {"url","source","tld","label","label_enc","class_final",
         "class_enc","fold","reg_domain"}

# Feature group taxonomy
FEATURE_GROUPS = {
    "Lexical":    ["url_len","num_dots","num_slashes","num_digits","num_specials",
                   "subdomain_len","domain_len","tld_len","digit_ratio","alpha_ratio",
                   "special_ratio","num_subdomains","has_www","subdomain_digits",
                   "path_len","path_depth","path_digits","has_php","has_exe","has_query",
                   "query_len","num_params","num_hyphens","num_underscores","num_at_signs",
                   "num_ampersands","num_equals","num_percent","has_at_in_url","has_port",
                   "has_ip","has_double_slash","url_entropy","domain_entropy","path_entropy",
                   "has_repeated_chars","has_hex_encoding","num_encoded_chars",
                   "has_double_dot","has_turkish_keyword","num_turkish_keywords",
                   "has_phishing_keyword","num_phishing_keywords","has_malware_keyword",
                   "num_malware_keywords","has_scam_keyword","num_scam_keywords",
                   "domain_hyphen_count","domain_digit_count","domain_has_number",
                   "is_suspicious_tld","is_free_hosting","url_len_bucket",
                   "num_tokens","max_token_len","mean_token_len"],
    "Adversarial":["contains_brand","brand_in_subdomain","brand_in_path",
                   "brand_not_in_domain","brand_tld_mismatch","num_brands_mentioned",
                   "brand_with_hyphen","brand_plus_keyword","min_brand_edit_dist",
                   "is_typo_squat","has_char_substitution","has_doubled_char",
                   "brand_homoglyph","excessive_subdomains","tr_in_subdomain",
                   "com_in_subdomain","brand_dot_in_subdomain","deep_subdomain_nesting",
                   "susp_tld_with_brand","susp_tld_with_keyword","short_domain_susp_tld",
                   "numeric_with_susp_tld","pct_encoded_ratio","has_unicode_escape",
                   "has_punycode","hex_in_domain","excessive_hyphens",
                   "random_looking_domain","consonant_heavy_domain","long_random_path",
                   "hash_like_segment","base64_like_segment","many_path_dirs",
                   "suspicious_file_in_path","has_url_in_url","has_redirect_param",
                   "at_symbol_trick","double_protocol"],
    "TR-Ling":    ["tr_char_count","tr_char_ratio","has_tr_char","tr_char_in_domain",
                   "tr_char_in_path","distinct_tr_chars","tr_stopword_count",
                   "tr_stopword_ratio","tr_suffix_count","vowel_harmony_score",
                   "tr_token_ratio","tr_bigram_score","tr_vs_en_bigram",
                   "langid_tr_confidence","is_turkish_dominant","has_tr_bank_term",
                   "has_tr_gov_term","has_tr_ecommerce_term","has_tr_telecom_term",
                   "tr_sector_count"],
    "TR-Ext":     ["tr_phishing_vocab_score","tr_transliteration_score",
                   "tr_brand_impersonation_score","tr_semantic_urgency_score",
                   "tr_scam_vocab_score"],
    "Graph":      ["rare_token_count","max_token_cluster_size","shared_token_degree",
                   "unique_token_ratio","token_reuse_score","domain_family_size",
                   "tld_token_cooccur","sibling_domain_count","domain_ngram_cluster",
                   "registrant_pattern_score","subdomain_reuse_count",
                   "path_template_reuse","host_pattern_degree","campaign_membership",
                   "token_centrality","is_hub_domain"],
}
GROUP_COLORS = {"Lexical":"#1565C0","Adversarial":"#6A1B9A",
                "TR-Ling":"#2E7D32","TR-Ext":"#00838F","Graph":"#B71C1C"}

plt.rcParams.update({
    "font.family":"serif","font.size":10,"axes.labelsize":10,
    "axes.titlesize":11,"legend.fontsize":8.5,"figure.dpi":150,
    "savefig.dpi":600,"pdf.fonttype":42,"ps.fonttype":42,
    "lines.linewidth":1.6,"axes.linewidth":0.8,
    "axes.spines.top":False,"axes.spines.right":False,
})

def save_fig(fig, name):
    fig.savefig(os.path.join(PNG_DIR,f"{name}.png"),dpi=600,bbox_inches="tight")
    fig.savefig(os.path.join(PDF_DIR,f"{name}.pdf"),bbox_inches="tight")
    plt.close(fig)

def feat_group(f):
    for g, feats in FEATURE_GROUPS.items():
        if f in feats: return g
    return "Other"

def group_color(f):
    return GROUP_COLORS.get(feat_group(f),"#607D8B")

# ── LOAD DATA ─────────────────────────────────────────────────
print("[1] Loading data ...")
df = pd.read_csv(INPUT_FILE, low_memory=False)
FEATURES = [c for c in df.columns if c not in META and c not in LEAKY]
FEATURES  = list(dict.fromkeys(FEATURES))
for c in FEATURES:
    if not pd.api.types.is_numeric_dtype(df[c]):
        df[c] = pd.to_numeric(df[c], errors="coerce")
X_all = df[FEATURES].fillna(0)
y_bin = df["label_enc"].values
y_5   = df["class_enc"].values
print(f"  {len(df):,} rows · {len(FEATURES)} features")

CLASS_NAMES_5 = (df[["class_enc","class_final"]].drop_duplicates()
                  .sort_values("class_enc")["class_final"].tolist())
CLASS_COLORS_5 = ["#1565C0","#B71C1C","#2E7D32","#F57F17","#6A1B9A"]

# ────────────────────────────────────────────────────────────
# FIG 1: CORRELATION HEATMAP  (top 30 by variance, CLEAN file)
# ────────────────────────────────────────────────────────────
print("[2] Correlation heatmap ...")
variances = X_all.var()
top30 = variances.nlargest(30).index.tolist()
corr  = X_all[top30].corr()

# Sort by hierarchical clustering for visual clarity
dist   = 1 - corr.abs()
dist   = (dist + dist.T) / 2
np.fill_diagonal(dist.values, 0)
link   = hierarchy.linkage(squareform(dist), method="average")
order  = hierarchy.leaves_list(link)
top30s = [top30[i] for i in order]
corr_s = corr.loc[top30s, top30s]

fig, ax = plt.subplots(figsize=(13, 11))
im = ax.imshow(corr_s.values, cmap="RdBu_r", vmin=-1, vmax=1, aspect="auto")
ax.set_xticks(range(30)); ax.set_yticks(range(30))
ax.set_xticklabels([f if len(f)<=22 else f[:19]+"..." for f in top30s],
                   rotation=45, ha="right", fontsize=7.5)
ax.set_yticklabels([f if len(f)<=22 else f[:19]+"..." for f in top30s], fontsize=7.5)
# Group-colour tick labels
for tick, feat in zip(ax.get_xticklabels(), top30s):
    tick.set_color(group_color(feat))
for tick, feat in zip(ax.get_yticklabels(), top30s):
    tick.set_color(group_color(feat))
# Annotate only cells with |r| > 0.6
for i in range(30):
    for j in range(30):
        v = corr_s.values[i,j]
        if abs(v) > 0.6 and i != j:
            ax.text(j,i,f"{v:.2f}",ha="center",va="center",
                    fontsize=5.5, color="white" if abs(v)>0.8 else "black")
cbar = fig.colorbar(im, ax=ax, fraction=0.03, pad=0.01)
cbar.set_label("Pearson $r$", fontsize=9)
ax.set_title("Feature Correlation Matrix — TUMC (top 30 by variance, "
             "hierarchically clustered)", fontweight="bold", fontsize=11)
# Group legend
handles = [mpatches.Patch(color=v,label=k) for k,v in GROUP_COLORS.items()]
ax.legend(handles=handles, fontsize=7.5, loc="lower right",
          bbox_to_anchor=(1.22,0), title="Group", title_fontsize=8)
fig.tight_layout()
save_fig(fig, "feature_correlation_heatmap")
print("  ✓ correlation heatmap")

# ────────────────────────────────────────────────────────────
# FIG 2: TOP-25 |PEARSON| WITH LABEL — binary AND 5class
# ────────────────────────────────────────────────────────────
print("[3] Label correlation plots ...")
fig, axes = plt.subplots(1, 2, figsize=(14, 8))
for ax, (y, label, title) in zip(axes, [
        (y_bin, "binary",  "Binary task"),
        (y_5,   "5class",  "5-class task (macro |r|)")]):
    if label == "binary":
        corr_lbl = X_all.corrwith(pd.Series(y, index=X_all.index)).abs()
    else:
        # macro: average |r| across 5 classes (OVR)
        corr_lbl = pd.Series(0.0, index=FEATURES)
        for ci in range(5):
            yci = (y == ci).astype(float)
            corr_lbl += X_all.corrwith(pd.Series(yci, index=X_all.index)).abs()
        corr_lbl /= 5
    top25 = corr_lbl.nlargest(25)
    colors = [group_color(f) for f in top25.index]
    ax.barh(range(25), top25.values[::-1], color=colors[::-1], alpha=0.88)
    ax.set_yticks(range(25))
    ax.set_yticklabels(top25.index[::-1], fontsize=8.5)
    ax.set_xlabel("|Pearson correlation| with class label", fontsize=10)
    ax.set_title(f"Top 25 Features — {title}", fontweight="bold", fontsize=11)
    for i, (f, v) in enumerate(zip(top25.index[::-1], top25.values[::-1])):
        ax.text(v+0.002, i, f"{v:.3f}", va="center", fontsize=7.5)
handles = [mpatches.Patch(color=v,label=k) for k,v in GROUP_COLORS.items()]
fig.legend(handles=handles, fontsize=8, title="Feature group",
           loc="lower center", ncol=5, bbox_to_anchor=(0.5,-0.04))
fig.suptitle("Top Features by |Pearson Correlation| with Class Label",
             fontweight="bold", fontsize=13)
fig.tight_layout(rect=[0,0.04,1,1])
save_fig(fig, "top_features_label_corr")
print("  ✓ label correlation")

# ────────────────────────────────────────────────────────────
# FIG 3: MUTUAL INFORMATION — binary + 5class
# ────────────────────────────────────────────────────────────
print("[4] Mutual information (sampled for speed) ...")
# Use 50k stratified sample for MI (full dataset is too slow)
rng = np.random.RandomState(SEED)
idx_mi = rng.choice(len(df), min(50000,len(df)), replace=False)
X_mi = X_all.values[idx_mi]

fig, axes = plt.subplots(1, 2, figsize=(14, 8))
for ax, (y_src, title) in zip(axes, [
        (y_bin[idx_mi], "Binary task"),
        (y_5[idx_mi],   "5-class task")]):
    mi = mutual_info_classif(X_mi, y_src, discrete_features=False,
                             random_state=SEED, n_neighbors=5)
    mi_s = pd.Series(mi, index=FEATURES)
    top25 = mi_s.nlargest(25)
    colors = [group_color(f) for f in top25.index]
    ax.barh(range(25), top25.values[::-1], color=colors[::-1], alpha=0.88)
    ax.set_yticks(range(25)); ax.set_yticklabels(top25.index[::-1], fontsize=8.5)
    ax.set_xlabel("Mutual information (nats)", fontsize=10)
    ax.set_title(f"Top 25 — {title}", fontweight="bold", fontsize=11)
    for i, v in enumerate(top25.values[::-1]):
        ax.text(v+0.001, i, f"{v:.3f}", va="center", fontsize=7.5)
fig.legend(handles=handles, fontsize=8, title="Feature group",
           loc="lower center", ncol=5, bbox_to_anchor=(0.5,-0.04))
fig.suptitle("Mutual Information with Class Label (50k stratified sample)",
             fontweight="bold", fontsize=13)
fig.tight_layout(rect=[0,0.04,1,1])
save_fig(fig, "mutual_information_scores")
print("  ✓ mutual information")

# Save MI series for convergence
mi_binary = mutual_info_classif(X_mi, y_bin[idx_mi], discrete_features=False,
                                random_state=SEED)
mi_binary = pd.Series(mi_binary, index=FEATURES)

# ────────────────────────────────────────────────────────────
# FIG 4: NEAR-ZERO VARIANCE + LOW MI  (problematic features)
# ────────────────────────────────────────────────────────────
print("[5] Near-zero variance analysis ...")
var_s    = X_all.var()
nzv_thr  = var_s.quantile(0.10)  # bottom 10% by variance
nzv_feats= var_s[var_s < nzv_thr].sort_values()

fig, axes = plt.subplots(1, 2, figsize=(13, 6))
# Left: near-zero variance
ax = axes[0]
n_show = min(20, len(nzv_feats))
colors  = [group_color(f) for f in nzv_feats.index[-n_show:]]
ax.barh(range(n_show), nzv_feats.values[-n_show:], color=colors, alpha=0.85)
ax.set_yticks(range(n_show))
ax.set_yticklabels(nzv_feats.index[-n_show:], fontsize=8.5)
ax.axvline(nzv_thr, color="red", ls="--", lw=1.2, label=f"10th pct={nzv_thr:.4f}")
ax.set_xlabel("Variance", fontsize=10)
ax.set_title(f"Near-Zero Variance Features (n={len(nzv_feats)})",
             fontweight="bold", fontsize=11)
ax.legend(fontsize=8)

# Right: variance vs MI scatter
ax2 = axes[1]
v_vals = var_s[FEATURES].values
m_vals = mi_binary[FEATURES].values
colors_sc = [group_color(f) for f in FEATURES]
ax2.scatter(v_vals, m_vals, c=colors_sc, alpha=0.5, s=20, rasterized=True)
# highlight near-zero variance
nzv_mask = var_s[FEATURES] < nzv_thr
ax2.scatter(v_vals[nzv_mask], m_vals[nzv_mask],
            c="red", alpha=0.8, s=40, zorder=5, label="Near-zero variance")
ax2.set_xlabel("Variance (log scale)", fontsize=10)
ax2.set_ylabel("Mutual information with label", fontsize=10)
ax2.set_xscale("log")
ax2.set_title("Variance vs Mutual Information", fontweight="bold", fontsize=11)
ax2.legend(handles=handles + [mpatches.Patch(color="red",label="Near-zero var")],
           fontsize=7, ncol=2)

fig.suptitle("Feature Quality: Near-Zero Variance and MI Scatter",
             fontweight="bold", fontsize=13)
fig.tight_layout()
save_fig(fig, "near_zero_variance")
print("  ✓ near-zero variance")

# ────────────────────────────────────────────────────────────
# FIG 5: MULTI-METHOD CONSENSUS RANKING
# (|corr| + MI + variance rank — 3-way comparison)
# ────────────────────────────────────────────────────────────
print("[6] Multi-method consensus ranking ...")
corr_lbl_bin = X_all.corrwith(pd.Series(y_bin, index=X_all.index)).abs()
rank_corr = corr_lbl_bin.rank(ascending=False)
rank_mi   = mi_binary.rank(ascending=False)
rank_var  = var_s.rank(ascending=False)
consensus = (rank_corr + rank_mi + rank_var) / 3
top_c     = consensus.nsmallest(25)

fig, axes = plt.subplots(1, 2, figsize=(14, 8))
# Left: consensus bar
ax = axes[0]
colors  = [group_color(f) for f in top_c.index]
ax.barh(range(25), (135 - top_c.values[::-1]), color=colors[::-1], alpha=0.88)
ax.set_yticks(range(25)); ax.set_yticklabels(top_c.index[::-1], fontsize=8.5)
ax.set_xlabel("Consensus score (higher = more important)", fontsize=10)
ax.set_title("Consensus Ranking: |Corr| + MI + Variance",
             fontweight="bold", fontsize=11)

# Right: rank comparison scatter (|corr| vs MI)
ax2 = axes[1]
top25_idx = top_c.index
for f in FEATURES:
    col = group_color(f)
    alpha = 0.7 if f in top25_idx else 0.15
    sz    = 60  if f in top25_idx else 12
    ax2.scatter(rank_corr[f], rank_mi[f], color=col, alpha=alpha,
                s=sz, rasterized=True)
for f in top25_idx:
    if rank_corr[f] < 30 or rank_mi[f] < 30:
        ax2.annotate(f[:18], (rank_corr[f], rank_mi[f]),
                     textcoords="offset points", xytext=(3,2), fontsize=5.5)
ax2.set_xlabel("|Corr| rank (lower = more important)", fontsize=10)
ax2.set_ylabel("MI rank", fontsize=10)
ax2.set_title("Rank Agreement: |Corr| vs MI (top-25 highlighted)",
              fontweight="bold", fontsize=11)
ax2.plot([0,135],[0,135],"k--",lw=0.7,alpha=0.4)

from scipy.stats import spearmanr
rho, pval = spearmanr(rank_corr, rank_mi)
ax2.text(0.05, 0.92, f"Spearman ρ = {rho:.3f}  p={pval:.2e}",
         transform=ax2.transAxes, fontsize=9,
         bbox=dict(boxstyle="round,pad=0.3", fc="white", ec="gray", alpha=0.8))

fig.legend(handles=handles, fontsize=8, title="Feature group",
           loc="lower center", ncol=5, bbox_to_anchor=(0.5,-0.04))
fig.suptitle("Multi-Method Feature Selection Consensus (Binary Task)",
             fontweight="bold", fontsize=13)
fig.tight_layout(rect=[0,0.04,1,1])
save_fig(fig, "multi_method_consensus")
print("  ✓ multi-method consensus")

# ────────────────────────────────────────────────────────────
# FIG 6: FEATURE STABILITY ACROSS 5 CV FOLDS
# (permutation importance per fold, mean ± std)
# ────────────────────────────────────────────────────────────
print("[7] Feature stability across folds (5 × perm importance) ...")
fold_perm = {f: [] for f in FEATURES}

for fid in range(5):
    tr = df[df["fold"]!=fid]; te = df[df["fold"]==fid]
    Xtr = tr[FEATURES].fillna(0).values; ytr = tr["class_enc"].values
    Xte = te[FEATURES].fillna(0).values; yte = te["class_enc"].values
    # Quick model (fewer trees for speed)
    m = HistGradientBoostingClassifier(max_depth=4, learning_rate=0.1,
                                        max_iter=100, random_state=SEED)
    m.fit(Xtr, ytr)
    # Subsample for speed
    rng2 = np.random.RandomState(SEED+fid)
    pidx = rng2.choice(len(Xte), min(3000,len(Xte)), replace=False)
    perm = permutation_importance(m, Xte[pidx], yte[pidx],
                                  n_repeats=3, random_state=SEED,
                                  scoring="f1_macro", n_jobs=-1)
    for i, f in enumerate(FEATURES):
        fold_perm[f].append(perm.importances_mean[i])
    print(f"  fold {fid} done")

perm_mean = pd.Series({f: np.mean(v) for f,v in fold_perm.items()})
perm_std  = pd.Series({f: np.std(v)  for f,v in fold_perm.items()})
top_perm  = perm_mean.nlargest(25)

fig, axes = plt.subplots(1, 2, figsize=(14, 8))
# Left: stability bar with error bars
ax = axes[0]
colors = [group_color(f) for f in top_perm.index]
ax.barh(range(25), top_perm.values[::-1],
        xerr=perm_std[top_perm.index].values[::-1],
        color=colors[::-1], alpha=0.85, capsize=3,
        error_kw={"elinewidth":1.2,"ecolor":"#333333"})
ax.set_yticks(range(25)); ax.set_yticklabels(top_perm.index[::-1], fontsize=8.5)
ax.set_xlabel("Permutation importance (macro-F1 drop ± std, 5 folds × 3 repeats)",
              fontsize=9)
ax.set_title("Feature Stability Across 5 CV Folds",
             fontweight="bold", fontsize=11)

# Right: CV of importance (std/mean = CV coefficient)
cv_coef = (perm_std[top_perm.index] / (perm_mean[top_perm.index].abs() + 1e-9))
ax2 = axes[1]
colors2 = [group_color(f) for f in cv_coef.index]
bars = ax2.barh(range(25), cv_coef.values[::-1], color=colors2[::-1], alpha=0.85)
ax2.set_yticks(range(25)); ax2.set_yticklabels(cv_coef.index[::-1], fontsize=8.5)
ax2.set_xlabel("Coefficient of variation (std / mean)", fontsize=10)
ax2.set_title("Importance Stability: Coefficient of Variation\n"
              "(lower = more stable across folds)", fontweight="bold", fontsize=11)
ax2.axvline(0.3, color="red", ls="--", lw=1, alpha=0.7, label="CV=0.3 threshold")
ax2.legend(fontsize=8)

fig.legend(handles=handles, fontsize=8, title="Feature group",
           loc="lower center", ncol=5, bbox_to_anchor=(0.5,-0.04))
fig.suptitle("Feature Importance Stability Across 5-Fold CV (5-Class HistGB)",
             fontweight="bold", fontsize=13)
fig.tight_layout(rect=[0,0.04,1,1])
save_fig(fig, "feature_stability_cv")
print("  ✓ feature stability")

# ────────────────────────────────────────────────────────────
# FIG 7: FEATURE SELECTION CURVE
# (F1 vs number of features, top-k by consensus)
# ────────────────────────────────────────────────────────────
print("[8] Feature selection curve ...")
sorted_feats = consensus.nsmallest(len(FEATURES)).index.tolist()
k_values     = [5,10,15,20,30,40,50,60,80,100,120,135]
f1_5c = []; f1_bin = []

tr = df[df["fold"]!=0]; te = df[df["fold"]==0]
Xtr_all = tr[FEATURES].fillna(0); Xte_all = te[FEATURES].fillna(0)

for k in k_values:
    feats_k = sorted_feats[:k]
    Xtr_k   = Xtr_all[feats_k].values
    Xte_k   = Xte_all[feats_k].values
    for target_col, f1_list in [("class_enc",f1_5c),("label_enc",f1_bin)]:
        m = HistGradientBoostingClassifier(max_depth=4, learning_rate=0.1,
                                            max_iter=100, random_state=SEED)
        m.fit(Xtr_k, tr[target_col].values)
        f1 = f1_score(te[target_col].values, m.predict(Xte_k),
                      average="macro", zero_division=0)
        f1_list.append(f1)
    print(f"  k={k} done")

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
for ax, (scores, task, full_score) in zip(axes, [
        (f1_5c, "5-class Macro-F1", 0.907),
        (f1_bin,"Binary Macro-F1",  0.975)]):
    ax.plot(k_values, scores, "o-", color="#1565C0", lw=2, ms=6, label="Top-k consensus")
    ax.axhline(full_score, color="#B71C1C", ls="--", lw=1.5,
               label=f"Full 135 features ({full_score:.3f})")
    # Mark the elbow (largest gain per feature)
    gains = np.diff(scores)
    elbow = k_values[np.argmax(gains)+1]
    ax.axvline(elbow, color="#2E7D32", ls=":", lw=1.5,
               label=f"Elbow (k={elbow})")
    ax.fill_between(k_values, scores, full_score,
                    alpha=0.08, color="#B71C1C",
                    label="Gap to full-feature score")
    ax.set_xlabel("Number of features (top-k by consensus rank)", fontsize=10)
    ax.set_ylabel(task, fontsize=10)
    ax.set_title(f"Feature Selection Curve — {task.split()[0]} Task",
                 fontweight="bold", fontsize=11)
    ax.legend(fontsize=8)
    ax.set_xlim(0, 140)
    lo = max(0, min(scores)-0.03)
    ax.set_ylim(lo, min(1.01, max(scores)+0.03))
    # Annotate key k values
    for kv, sv in zip(k_values, scores):
        if kv in [10, 30, 50, 135]:
            ax.annotate(f"{sv:.3f}", (kv, sv),
                        textcoords="offset points", xytext=(3,6), fontsize=7.5)

fig.suptitle("Feature Selection Curve: Performance vs Number of Features",
             fontweight="bold", fontsize=13)
fig.tight_layout()
save_fig(fig, "feature_selection_curve")
print("  ✓ feature selection curve")

# ────────────────────────────────────────────────────────────
# FIG 8: FEATURE GROUP BREAKDOWN  (count + mean |corr| + MI)
# ────────────────────────────────────────────────────────────
print("[9] Feature group breakdown ...")
grp_data = {}
for grp, feats in FEATURE_GROUPS.items():
    present = [f for f in feats if f in FEATURES]
    grp_data[grp] = {
        "n":         len(present),
        "mean_corr": corr_lbl_bin[present].mean() if present else 0,
        "mean_mi":   mi_binary[present].mean()    if present else 0,
        "mean_perm": perm_mean[present].mean()    if present else 0,
    }
grp_df = pd.DataFrame(grp_data).T

fig = plt.figure(figsize=(13, 5))
spec = gridspec.GridSpec(1, 4, figure=fig, wspace=0.35)
axes = [fig.add_subplot(spec[0,i]) for i in range(4)]
grp_names  = list(FEATURE_GROUPS.keys())
grp_cols   = [GROUP_COLORS[g] for g in grp_names]

for ax, (col, label) in zip(axes, [
        ("n",         "Feature count"),
        ("mean_corr", "Mean |Pearson corr|"),
        ("mean_mi",   "Mean mutual information"),
        ("mean_perm", "Mean perm importance"),
]):
    vals = [grp_df.loc[g, col] for g in grp_names]
    bars = ax.bar(range(len(grp_names)), vals, color=grp_cols, alpha=0.85)
    ax.set_xticks(range(len(grp_names)))
    ax.set_xticklabels([g.replace("-","\n") for g in grp_names],
                       fontsize=8, rotation=0)
    ax.set_ylabel(label, fontsize=9)
    ax.set_title(label, fontweight="bold", fontsize=9)
    for bar, v in zip(bars, vals):
        ax.text(bar.get_x()+bar.get_width()/2,
                bar.get_height()*1.02, f"{v:.3f}" if col!="n" else str(int(v)),
                ha="center", va="bottom", fontsize=8)

fig.suptitle("Feature Group Summary: Count, Correlation, MI, and Permutation Importance",
             fontweight="bold", fontsize=12)
save_fig(fig, "feature_group_summary")
print("  ✓ feature group breakdown")

# ────────────────────────────────────────────────────────────
# FIG 9: PAIRWISE REDUNDANCY (|r| > 0.70 pairs annotated)
# ────────────────────────────────────────────────────────────
print("[10] Pairwise redundancy analysis ...")
# Build list of highly correlated pairs
corr_all = X_all[FEATURES].corr().abs()
np.fill_diagonal(corr_all.values, 0)
thresh = 0.70
pairs  = []
seen   = set()
for i, f1 in enumerate(FEATURES):
    for j, f2 in enumerate(FEATURES):
        if j <= i: continue
        r = corr_all.loc[f1, f2]
        if r >= thresh:
            key = tuple(sorted([f1,f2]))
            if key not in seen:
                pairs.append((f1, f2, r))
                seen.add(key)
pairs.sort(key=lambda x: -x[2])
print(f"    Pairs |r|≥{thresh}: {len(pairs)}")

fig, axes = plt.subplots(1, 2, figsize=(14, 7))
# Left: top-20 redundant pairs bar
ax = axes[0]
n_show = min(20, len(pairs))
pair_labels = [f"{p[0][:18]}\n↔ {p[1][:18]}" for p in pairs[:n_show]]
pair_vals   = [p[2] for p in pairs[:n_show]]
pair_cols   = [GROUP_COLORS.get(feat_group(p[0]),"#607D8B") for p in pairs[:n_show]]
ax.barh(range(n_show), pair_vals[::-1], color=pair_cols[::-1], alpha=0.85)
ax.set_yticks(range(n_show)); ax.set_yticklabels(pair_labels[::-1], fontsize=7.5)
ax.axvline(thresh, color="red", ls="--", lw=1.2, label=f"|r|={thresh} threshold")
ax.set_xlabel("|Pearson r|", fontsize=10)
ax.set_title(f"Top-{n_show} Redundant Feature Pairs (|r| ≥ {thresh})",
             fontweight="bold", fontsize=11)
ax.legend(fontsize=8)

# Right: redundancy histogram
ax2 = axes[1]
flat_corr = corr_all.values[np.triu_indices_from(corr_all.values, k=1)]
ax2.hist(flat_corr[flat_corr > 0.05], bins=50,
         color="#1565C0", alpha=0.75, edgecolor="white", linewidth=0.4)
ax2.axvline(thresh, color="red", ls="--", lw=1.5,
            label=f"Redundancy threshold ({thresh})")
ax2.axvline(0.95, color="#B71C1C", ls=":", lw=1.5, label="|r|=0.95 high-corr")
ax2.set_xlabel("|Pearson r| between feature pairs", fontsize=10)
ax2.set_ylabel("Count of feature pairs", fontsize=10)
ax2.set_title(f"Pairwise |r| Distribution (n={len(flat_corr):,} pairs)",
              fontweight="bold", fontsize=11)
ax2.legend(fontsize=8)
ax2.text(0.97, 0.97,
         f"Pairs |r|≥{thresh}: {len(pairs)}\n"
         f"Pairs |r|≥0.95: {sum(1 for p in pairs if p[2]>=0.95)}",
         transform=ax2.transAxes, ha="right", va="top", fontsize=8.5,
         bbox=dict(boxstyle="round,pad=0.3",fc="white",ec="gray",alpha=0.8))

fig.suptitle("Feature Redundancy Analysis — Pairwise Pearson |r|",
             fontweight="bold", fontsize=13)
fig.tight_layout()
save_fig(fig, "feature_redundancy")
print("  ✓ pairwise redundancy")

# ────────────────────────────────────────────────────────────
# FIG 10: CLASS-CONDITIONAL VIOLIN PLOTS  (top-6 features)
# ────────────────────────────────────────────────────────────
print("[11] Class-conditional violin plots ...")
top6 = consensus.nsmallest(6).index.tolist()
# Sample for plotting
samp = df.sample(min(20000,len(df)), random_state=SEED)
X_samp = samp[FEATURES].fillna(0)
y_samp = samp["class_enc"].values
cname_map = dict(zip(range(5), CLASS_NAMES_5))

fig, axes = plt.subplots(2, 3, figsize=(14, 8))
for ax, feat in zip(axes.flat, top6):
    vals_by_class = [X_samp[feat][y_samp==ci].values for ci in range(5)]
    vp = ax.violinplot(vals_by_class, positions=range(5),
                       showmedians=True, showextrema=False)
    for ci, (body, color) in enumerate(zip(vp["bodies"], CLASS_COLORS_5)):
        body.set_facecolor(color); body.set_alpha(0.6)
    vp["cmedians"].set_color("black"); vp["cmedians"].set_linewidth(1.5)
    ax.set_xticks(range(5))
    ax.set_xticklabels([c[:8] for c in CLASS_NAMES_5],
                       rotation=25, ha="right", fontsize=8.5)
    ax.set_title(feat[:30], fontweight="bold", fontsize=9)
    ax.set_ylabel("Feature value", fontsize=8)
    grp = feat_group(feat)
    ax.text(0.97, 0.97, f"[{grp}]", transform=ax.transAxes,
            ha="right", va="top", fontsize=7.5,
            color=GROUP_COLORS.get(grp,"#607D8B"))
fig.suptitle("Class-Conditional Feature Distributions — Top-6 by Consensus Rank\n"
             "(violin = distribution density; line = median)",
             fontweight="bold", fontsize=12)
fig.tight_layout()
save_fig(fig, "class_conditional_distributions")
print("  ✓ class-conditional distributions")

# ────────────────────────────────────────────────────────────
# FIG 11: COMPREHENSIVE SELECTION SUMMARY (LaTeX + final plot)
# ────────────────────────────────────────────────────────────
print("[12] Summary table ...")
top20 = consensus.nsmallest(20).index.tolist()
summary = pd.DataFrame({
    "Feature":      top20,
    "Group":        [feat_group(f)      for f in top20],
    "|Corr| rank":  [int(rank_corr[f])  for f in top20],
    "MI rank":      [int(rank_mi[f])    for f in top20],
    "Perm rank":    [int(perm_mean.rank(ascending=False)[f]) for f in top20],
    "Consensus":    [round(consensus[f],1) for f in top20],
})
summary["Variance"] = [round(float(var_s[f]),4) for f in top20]

tex_lines = [
    r"\begin{table}[htbp]", r"\centering",
    r"\caption{Top-20 features by three-method consensus ranking "
    r"(|Pearson $r$|, mutual information, permutation importance). "
    r"All ranks are on a 1--135 scale (lower = more important). "
    r"Consensus = mean of three ranks; features with low consensus "
    r"and low variance are candidates for pruning in latency-critical "
    r"deployment scenarios.}",
    r"\label{tab:feature_selection_summary}",
    r"\footnotesize",
    r"\begin{tabular}{llrrrrl}",
    r"\toprule",
    r"Feature & Group & $|r|$ rank & MI rank & Perm rank & Consensus & Variance \\",
    r"\midrule",
]
for _, row in summary.iterrows():
    tex_lines.append(
        f"{row['Feature']} & {row['Group']} & {row['|Corr| rank']} & "
        f"{row['MI rank']} & {row['Perm rank']} & {row['Consensus']:.1f} & "
        f"{row['Variance']} \\\\")
tex_lines += [r"\bottomrule", r"\end{tabular}", r"\end{table}"]
with open(os.path.join(TEX_DIR,"feature_selection_summary.tex"),"w") as f:
    f.write("\n".join(tex_lines))
print("  ✓ LaTeX summary table")

# ────────────────────────────────────────────────────────────
# PRINT FINAL REPORT
# ────────────────────────────────────────────────────────────
print(f"""
{'='*60}
FEATURE SELECTION PLOTS — COMPLETE
{'='*60}
Output directory: {OUT_DIR}

Figures generated (12):
  feature_correlation_heatmap.png   ← replaces old polluted version
  top_features_label_corr.png       ← binary + 5class side-by-side
  mutual_information_scores.png     ← MI (50k sample) both tasks
  near_zero_variance.png            ← NZV + MI scatter
  multi_method_consensus.png        ← |corr| + MI + variance ranking
  feature_stability_cv.png          ← perm importance ± std, 5 folds
  feature_selection_curve.png       ← F1 vs top-k features
  feature_group_summary.png         ← 4-panel group breakdown
  feature_redundancy.png            ← pairs |r|≥0.70 + histogram
  class_conditional_distributions.png ← violin plots, top-6 features
  (PDF copies in figures_pdf/)

LaTeX table:
  tex/feature_selection_summary.tex  ← \\input{{}} ready

Key findings:
  Total features: {len(FEATURES)}
  Near-zero variance (bottom 10%): {len(nzv_feats)}
  Redundant pairs |r|≥0.70: {len(pairs)}
  Elbow (5-class): k={k_values[np.argmax(np.diff(f1_5c))+1]} features
{'='*60}
""")

[1] Loading data ...
  1,239,308 rows · 135 features
[2] Correlation heatmap ...
  ✓ correlation heatmap
[3] Label correlation plots ...
  ✓ label correlation
[4] Mutual information (sampled for speed) ...
  ✓ mutual information
[5] Near-zero variance analysis ...
  ✓ near-zero variance
[6] Multi-method consensus ranking ...
  ✓ multi-method consensus
[7] Feature stability across folds (5 × perm importance) ...
  fold 0 done
  fold 1 done
  fold 2 done
  fold 3 done
  fold 4 done
  ✓ feature stability
[8] Feature selection curve ...
  k=5 done
  k=10 done
  k=15 done
  k=20 done
  k=30 done
  k=40 done
  k=50 done
  k=60 done
  k=80 done
  k=100 done
  k=120 done
  k=135 done
  ✓ feature selection curve
[9] Feature group breakdown ...
  ✓ feature group breakdown
[10] Pairwise redundancy analysis ...
    Pairs |r|≥0.7: 91
  ✓ pairwise redundancy
[11] Class-conditional violin plots ...
  ✓ class-conditional distributions
[12] Summary table ...
  ✓ LaTeX summary table

FEATURE SELECTION P